# Visualización de una simulación TVB de una región

Este cuaderno carga el contenedor binario NumPy generado por `src/run_simulation.py` y representa el estímulo, las tasas de disparo y las variables de adaptación.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

candidates = [Path('results/simulation.npz'), Path('../results/simulation.npz')]
result_path = next((path for path in candidates if path.exists()), None)
if result_path is None:
    raise FileNotFoundError(
        'Ejecuta primero: python src/run_simulation.py --config config/simulation.json'
    )

with np.load(result_path, allow_pickle=False) as stored:
    result = {name: stored[name].copy() for name in stored.files if name != 'metadata_json'}
    metadata = json.loads(str(stored['metadata_json']))

print(f'Fichero: {result_path.resolve()}')
print(f"Modelo: {metadata['model']} | regiones: {metadata['region_count']}")
print(f"Estímulo: {metadata['config']['stimulus']['type']}")

In [ ]:
time_s = result['time_ms'] / 1000.0
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True, constrained_layout=True)

axes[0].plot(time_s, result['stimulus_hz'], color='black', label='Estímulo base')
axes[0].plot(time_s, result['Fe_ext_hz'], label='Fe externa', alpha=0.85)
axes[0].plot(time_s, result['Fi_ext_hz'], label='Fi externa', alpha=0.85)
axes[0].set_ylabel('Entrada (Hz)')
axes[0].legend(loc='upper right')

axes[1].plot(time_s, result['E_hz'], label='E', color='tab:blue')
axes[1].plot(time_s, result['I_hz'], label='I', color='tab:orange')
axes[1].set_ylabel('Tasa (Hz)')
axes[1].legend(loc='upper right')

axes[2].plot(time_s, result['W_e_pA'], label='W_e', color='tab:green')
axes[2].plot(time_s, result['W_i_pA'], label='W_i', color='tab:red')
axes[2].set_ylabel('Adaptación (pA)')
axes[2].set_xlabel('Tiempo (s)')
axes[2].legend(loc='upper right')

for axis in axes:
    axis.grid(alpha=0.25)

fig.suptitle('Zerlaut-AdEx en una única región de TVB')
plt.show()

In [ ]:
summary = {
    'duración (s)': float(time_s[-1]),
    'E media (Hz)': float(np.mean(result['E_hz'])),
    'E máxima (Hz)': float(np.max(result['E_hz'])),
    'I media (Hz)': float(np.mean(result['I_hz'])),
    'I máxima (Hz)': float(np.max(result['I_hz'])),
}
summary